# Qwen3 Embedding 0.6B INT4 AWQ

This notebook clones `awq-embed`, installs the quantization-only runtime, runs AWQ search on local dummy calibration text, and saves packed INT4 weights for `Qwen/Qwen3-Embedding-0.6B`.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable a GPU runtime before running AWQ quantization.")
print("gpu", torch.cuda.get_device_name(0))
print("bf16 supported", torch.cuda.is_bf16_supported())

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d awq-embed ] || git clone https://github.com/phunggiahuy159/awq-embed.git
%cd /content/awq-embed

!python -m pip install --upgrade pip
!python -m pip install -e . --no-deps
!python -m pip install "cachetools<7" "transformers>=4.51.0" "accelerate==0.34.2" datasets sentencepiece "tokenizers>=0.12.1" texttable toml attributedict protobuf tqdm scikit-learn scipy

In [ ]:
%%bash
python - <<'PY'
import awq.entry
import awq.quantize.pre_quant
import awq.quantize.auto_scale
import awq.quantize.qmodule
print("AWQ imports succeeded without compiled kernels")
PY

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --calib_data dummy \
  --calib_n_samples 16 \
  --calib_seqlen 128 \
  --run_awq \
  --dump_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --load_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt \
  --q_backend real \
  --dump_quant quant_cache/qwen3-embedding-0.6b-w4-g128-awq.pt

In [ ]:
from pathlib import Path

paths = [
    Path("awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt"),
    Path("quant_cache/qwen3-embedding-0.6b-w4-g128-awq-v2.pt"),
]
for path in paths:
    if path.exists():
        print(f"{path}: {path.stat().st_size / (1024 ** 2):.2f} MiB")
    else:
        print(f"missing: {path}")

In [ ]:
# Fast MTEB-style classification probe: original vs AWQ pseudo-INT4.
# The packed INT4 .pt checkpoint is for kernel-backed inference; this cell uses
# fake/pseudo quantization so it can run in plain Colab PyTorch without building kernels.
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from transformers import AutoModel, AutoTokenizer

from awq.quantize.pre_quant import apply_awq
from awq.quantize.quantizer import pseudo_quantize_model_weight

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
AWQ_PATH = Path("awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", trust_remote_code=True)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("ag_news")
train_texts = dataset["train"]["text"][:512]
train_labels = dataset["train"]["label"][:512]
test_texts = dataset["test"]["text"][:256]
test_labels = dataset["test"]["label"][:256]

def last_token_pool(last_hidden_states, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

@torch.no_grad()
def encode_texts(model, texts, batch_size=16, max_length=256):
    embeddings = []
    model.eval()
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(DEVICE)
        outputs = model(**encoded)
        pooled = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
        embeddings.append(pooled.float().cpu().numpy())
    return np.concatenate(embeddings, axis=0)

def eval_embeddings(name, model):
    x_train = encode_texts(model, train_texts)
    x_test = encode_texts(model, test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(x_train, train_labels)
    pred = clf.predict(x_test)
    acc = accuracy_score(test_labels, pred)
    print(f"{name} AG News accuracy: {acc:.4f}")
    return acc

original_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
original_acc = eval_embeddings("Original", original_model)
del original_model
gc.collect()
torch.cuda.empty_cache()

awq_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to("cpu")
awq_results = torch.load(AWQ_PATH, map_location="cpu")
apply_awq(awq_model, awq_results)
pseudo_quantize_model_weight(
    awq_model,
    w_bit=4,
    q_config={"zero_point": True, "q_group_size": 128},
)
awq_model = awq_model.to(DEVICE)
awq_acc = eval_embeddings("AWQ pseudo-INT4", awq_model)
print(f"Accuracy delta: {awq_acc - original_acc:+.4f}")


In [ ]:
# Broader fast classification comparison.
# Run the previous eval cell first so `awq_model`, `tokenizer`, `DEVICE`, and `DTYPE` exist.
# This reports original vs AWQ fake-INT4 quality. The packed INT4 checkpoint still
# requires compiled AWQ kernels for direct packed-weight inference.
import gc

TASKS = [
    {
        "name": "AG News",
        "loader": ("ag_news", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "Emotion",
        "loader": ("emotion", None),
        "train_split": "train",
        "test_split": "validation",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "TREC coarse",
        "loader": ("CogComp/trec", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "coarse_label",
    },
    {
        "name": "Rotten Tomatoes",
        "loader": ("cornell-movie-review-data/rotten_tomatoes", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "TweetEval sentiment",
        "loader": ("cardiffnlp/tweet_eval", "sentiment"),
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "DBpedia 14",
        "loader": ("fancyzhx/dbpedia_14", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "content",
        "label_col": "label",
    },
    {
        "name": "SST-2",
        "loader": ("glue", "sst2"),
        "train_split": "train",
        "test_split": "validation",
        "text_col": "sentence",
        "label_col": "label",
    },
    {
        "name": "Yelp Polarity",
        "loader": ("fancyzhx/yelp_polarity", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "text",
        "label_col": "label",
    },
    {
        "name": "Amazon Polarity",
        "loader": ("fancyzhx/amazon_polarity", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "content",
        "label_col": "label",
    },
    {
        "name": "Yahoo Answers Topics",
        "loader": ("fancyzhx/yahoo_answers_topics", None),
        "train_split": "train",
        "test_split": "test",
        "text_col": "question_content",
        "label_col": "topic",
    },
]

def take_examples(split, text_col, label_col, limit):
    limit = min(limit, len(split))
    return split[text_col][:limit], split[label_col][:limit]

def score_task(model, task, train_limit=512, test_limit=256):
    dataset_name, config_name = task["loader"]
    ds = load_dataset(dataset_name, config_name) if config_name else load_dataset(dataset_name)
    train_texts, train_labels = take_examples(ds[task["train_split"]], task["text_col"], task["label_col"], train_limit)
    test_texts, test_labels = take_examples(ds[task["test_split"]], task["text_col"], task["label_col"], test_limit)
    x_train = encode_texts(model, train_texts)
    x_test = encode_texts(model, test_texts)
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(x_train, train_labels)
    return accuracy_score(test_labels, clf.predict(x_test))

original_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)
rows = []
for task in TASKS:
    try:
        original_score = score_task(original_model, task)
        awq_score = score_task(awq_model, task)
        rows.append({
            "task": task["name"],
            "original": original_score,
            "awq_fake_int4": awq_score,
            "delta": awq_score - original_score,
        })
        print(f"{task['name']}: original={original_score:.4f} awq_fake_int4={awq_score:.4f} delta={awq_score - original_score:+.4f}")
    except Exception as exc:
        print(f"Skipping {task['name']}: {type(exc).__name__}: {exc}")

if not rows:
    raise RuntimeError("No classification tasks completed successfully.")
mean_original = sum(row["original"] for row in rows) / len(rows)
mean_awq = sum(row["awq_fake_int4"] for row in rows) / len(rows)
mean_delta = mean_awq - mean_original

print("\nFinal comparison")
print("| Task | Original | AWQ fake-INT4 | Delta |")
print("|---|---:|---:|---:|")
for row in rows:
    print(f"| {row['task']} | {row['original']:.4f} | {row['awq_fake_int4']:.4f} | {row['delta']:+.4f} |")
print(f"| **Average** | **{mean_original:.4f}** | **{mean_awq:.4f}** | **{mean_delta:+.4f}** |")

del original_model
gc.collect()
torch.cuda.empty_cache()
rows


In [ ]:
# Pair-task probes: STS, paraphrase, and NLI datasets.
# Run after the first eval cell so `awq_model`, `encode_texts`, `MODEL_ID`, `DEVICE`, and `DTYPE` exist.
import gc

import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from transformers import AutoModel

def cosine_scores(model, left_texts, right_texts):
    left = encode_texts(model, left_texts)
    right = encode_texts(model, right_texts)
    return (left * right).sum(axis=1)

def score_similarity(model, dataset_name, config_name, split, left_col, right_col, label_col, limit=512):
    ds = load_dataset(dataset_name, config_name, split=split) if config_name else load_dataset(dataset_name, split=split)
    limit = min(limit, len(ds))
    scores = cosine_scores(model, ds[left_col][:limit], ds[right_col][:limit])
    labels = np.asarray(ds[label_col][:limit], dtype=np.float32)
    return float(spearmanr(scores, labels).correlation)

def pair_features(model, premises, hypotheses):
    premise_emb = encode_texts(model, premises)
    hypothesis_emb = encode_texts(model, hypotheses)
    return np.concatenate(
        [premise_emb, hypothesis_emb, np.abs(premise_emb - hypothesis_emb), premise_emb * hypothesis_emb],
        axis=1,
    )

def score_pair_classification(model, dataset_name, config_name, train_split, test_split, left_col, right_col, label_col, train_limit=512, test_limit=256):
    train = load_dataset(dataset_name, config_name, split=train_split) if config_name else load_dataset(dataset_name, split=train_split)
    test = load_dataset(dataset_name, config_name, split=test_split) if config_name else load_dataset(dataset_name, split=test_split)
    train_limit = min(train_limit, len(train))
    test_limit = min(test_limit, len(test))
    x_train = pair_features(model, train[left_col][:train_limit], train[right_col][:train_limit])
    x_test = pair_features(model, test[left_col][:test_limit], test[right_col][:test_limit])
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(x_train, train[label_col][:train_limit])
    return accuracy_score(test[label_col][:test_limit], clf.predict(x_test))

PAIR_TASKS = [
    {
        "name": "STS-B",
        "metric": "Spearman",
        "kind": "similarity",
        "args": ("glue", "stsb", "validation", "sentence1", "sentence2", "label"),
    },
    {
        "name": "MRPC",
        "metric": "Accuracy",
        "kind": "classification",
        "args": ("glue", "mrpc", "train", "validation", "sentence1", "sentence2", "label"),
    },
    {
        "name": "QQP",
        "metric": "Accuracy",
        "kind": "classification",
        "args": ("glue", "qqp", "train", "validation", "question1", "question2", "label"),
    },
    {
        "name": "MNLI matched",
        "metric": "Accuracy",
        "kind": "classification",
        "args": ("glue", "mnli", "train", "validation_matched", "premise", "hypothesis", "label"),
    },
    {
        "name": "QNLI",
        "metric": "Accuracy",
        "kind": "classification",
        "args": ("glue", "qnli", "train", "validation", "question", "sentence", "label"),
    },
    {
        "name": "RTE",
        "metric": "Accuracy",
        "kind": "classification",
        "args": ("glue", "rte", "train", "validation", "sentence1", "sentence2", "label"),
    },
]

original_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True).to(DEVICE)

pair_rows = []
for task in PAIR_TASKS:
    try:
        if task["kind"] == "similarity":
            original_score = score_similarity(original_model, *task["args"])
            awq_score = score_similarity(awq_model, *task["args"])
        else:
            original_score = score_pair_classification(original_model, *task["args"])
            awq_score = score_pair_classification(awq_model, *task["args"])
        pair_rows.append({
            "task": task["name"],
            "metric": task["metric"],
            "original": original_score,
            "awq_fake_int4": awq_score,
            "delta": awq_score - original_score,
        })
        print(f"{task['name']} {task['metric']}: original={original_score:.4f} awq_fake_int4={awq_score:.4f} delta={awq_score - original_score:+.4f}")
    except Exception as exc:
        print(f"Skipping {task['name']}: {type(exc).__name__}: {exc}")

if not pair_rows:
    raise RuntimeError("No pair tasks completed successfully.")

print("\nPair-task final comparison")
print("| Task | Metric | Original | AWQ fake-INT4 | Delta |")
print("|---|---|---:|---:|---:|")
for row in pair_rows:
    print(f"| {row['task']} | {row['metric']} | {row['original']:.4f} | {row['awq_fake_int4']:.4f} | {row['delta']:+.4f} |")

del original_model
gc.collect()
torch.cuda.empty_cache()
pair_rows
